# Lab 2 Report: 
## Iris Classification with Regression

### Name:

In [ ]:
# Import neccessary packages

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

import torch


In [ ]:
from IPython.display import Image # For displaying images in colab jupyter cell

In [ ]:
Image('lab2_exercise1.PNG', width = 1000)

## Prepare Data

In [196]:
from sklearn.datasets import load_iris

# iris dataset is available from scikit-learn package
iris = load_iris()

# Load the X (features) and y (targets) for training
X_train = iris['data']
y_train = iris['target']
y_train = y_train[:, np.newaxis] # Reshape y_train to be a column vector instead of 1D array


# Load the name labels for features and targets
feature_names = iris['feature_names']
names = iris['target_names']

# Feel free to perform additional data processing here (e.g. standard scaling)

In [ ]:
def scale_data(arr):
    mean = np.mean(arr, axis = 0)
    std = np.std(arr, axis = 0)
    z = (arr - mean) / std
    return z

In [ ]:
#X_train_scaled = scale_data(X_train)
#y_train_scaled = scale_data(y_train)

#plt.hist(X_train_scaled, bins = 20)
#plt.show()
#plt.hist(X_train, bins = 20)
#plt.show()
#plt.hist(y_train_scaled, bins = 20)
#plt.show()

In [ ]:
# Print the first 10 training samples for both features and targets

print(X_train[:10, :], y_train[:10]) 

In [ ]:
# Print the dimensions of features and targets

print(X_train.shape, y_train.shape)

In [ ]:
# feature_names contains name for each column in X_train
# For targets, 0 -> setosa, 1 -> versicolor, 2 -> virginica

print(feature_names, names)

In [ ]:
# We can visualize the dataset before training

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# enumerate picks up both the index (0, 1, 2) and the element ('setosa', 'versicolor', 'virginica') from "names"
# loop 1: target = 0, target_name = 'setosa'
# loop 2: target = 1, target_name = 'versicolor' etc

for target, target_name in enumerate(names):
    
    # Subset the rows of X_train that fall into each flower category using boolean mapping
    X_plot = X_train[y_train.ravel() == target]
    
    # Plot the sepal length versus sepal width for the flower category
    ax1.plot(X_plot[:, 0], X_plot[:, 1], linestyle='none', marker='o', label=target_name)

# Label the plot
ax1.set_xlabel(feature_names[0])
ax1.set_ylabel(feature_names[1])
ax1.axis('equal')
ax1.legend()

# Repeat the above process but with petal length versus petal width
for target, target_name in enumerate(names):
    
    X_plot = X_train[y_train.ravel() == target]
    
    ax2.plot(X_plot[:, 2], X_plot[:, 3], linestyle='none', marker='o', label=target_name)
    
ax2.set_xlabel(feature_names[2])
ax2.set_ylabel(feature_names[3])
ax2.axis('equal')
ax2.legend()

plt.show()

## Define Model

In [197]:
class irisClassification(torch.nn.Module):
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        
        super(irisClassification, self).__init__()
        
        self.layer1 = torch.nn.Linear(input_dim, hidden_dim) 
        self.layer2 = torch.nn.Linear(hidden_dim, output_dim) 
        self.activation = torch.nn.ReLU()
        
    def forward(self, x):
        
        out = self.layer1(x)
        out = self.activation(out)
        out = self.layer2(out)
        
        return out

## Define Hyperparameters

In [198]:
model = irisClassification(input_dim = 4, hidden_dim = 10, output_dim = 1)

learning_rate = 0.01
epochs  = 30

# We will use gradient descent for our optimizer and Mean Squared Error Loss function
loss_func = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

## Identify Tracked Values

In [199]:
# follow models performance over each epoch. Identify a metric and track it over epochs

training_loss = []

## Train Model

In [200]:

X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).long().float()

for epoch in range(epochs):

    optimizer.zero_grad() # Empty the gradient buffer so each learning event per epoch is separate
    outputs = model(X_train) # Forward pass the inputs through the network to produce outputs
    loss = loss_func(outputs.squeeze(), y_train) # Compute the loss via comparing the output with expected targets

    training_loss.append(loss.item()) # Save the loss value to training_loss we defined
    optimizer.zero_grad() # Empty the gradient buffer so each learning event per epoch is separate
    loss.backward() # Compute how much changes to be made to weights/biases

    optimizer.step() # Update the weights/biases according to learning rate
    training_loss.append(loss.item()) # Save the loss value to training_loss we defined

    print('epoch {}, loss {}'.format(epoch, loss.item()))

epoch 0, loss 0.681964099407196
epoch 1, loss 0.6806055307388306
epoch 2, loss 0.6802915334701538
epoch 3, loss 0.6801841259002686
epoch 4, loss 0.6801190376281738
epoch 5, loss 0.6800633072853088
epoch 6, loss 0.6800100803375244
epoch 7, loss 0.679957926273346
epoch 8, loss 0.6799064874649048
epoch 9, loss 0.6798558831214905
epoch 10, loss 0.6798058748245239
epoch 11, loss 0.6797565817832947
epoch 12, loss 0.6797080039978027
epoch 13, loss 0.6796601414680481
epoch 14, loss 0.679612934589386
epoch 15, loss 0.6795662045478821
epoch 16, loss 0.679520308971405
epoch 17, loss 0.679474949836731
epoch 18, loss 0.6794303059577942
epoch 19, loss 0.6793862581253052
epoch 20, loss 0.6793428659439087
epoch 21, loss 0.6792999505996704
epoch 22, loss 0.6792577505111694
epoch 23, loss 0.6792160272598267
epoch 24, loss 0.6791749000549316
epoch 25, loss 0.6791344285011292
epoch 26, loss 0.6790945529937744
epoch 27, loss 0.6790550351142883
epoch 28, loss 0.6790161728858948
epoch 29, loss 0.678977966308

## Visualize and Evaluate Model

In [ ]:
# Plot your training loss throughout the training
# Include proper x and y labels for the plot

plt.figure(figsize=(12, 6))
plt.plot(training_loss, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.legend()
plt.show()

# YOUR CODE HERE

In [201]:
# Confirm that your model's training accuracy is >90%

with torch.no_grad():
    outputs = model(X_train)
    predicted = outputs.detach().numpy()
    # Convert predictions and targets to label indices (0,1,2) and ensure same type
    pred_labels = predicted.argmax(axis=1)
    true_labels = y_train.argmax(dim=1).numpy()  # y_train is a torch one-hot tensor
    # Compare predictions with targets to compute the training accuracy
    correct_predictions = (pred_labels == true_labels).sum()
    training_accuracy = correct_predictions / true_labels.shape[0]
    print('Training accuracy: {:.2f}%'.format(training_accuracy * 100))

# Training accuracy = (# of correct predictions) / (total # of training samples)
# You can round the model predictions to integer (e.g. 0.34 -> 0, 1.78 -> 2)

# YOUR CODE HERE

Training accuracy: 100.00%
